# 📱 Pocket OTC AI Analyzer — APK Builder

يبني APK مباشرة من GitHub داخل Google Colab بدون Telegram وبدون GitHub Actions.

READ-ONLY: التطبيق لا يسجل الدخول إلى Pocket Option ولا ينفذ صفقات.

In [ ]:
# 🚀 بناء APK بنقرة واحدة
import os, shutil, subprocess, traceback, zipfile, stat

REPO='https://github.com/mohmb142/Jjjjjjj.git'
ZIP_URL='https://github.com/mohmb142/Jjjjjjj/archive/refs/heads/main.zip'
ROOT='/content/Jjjjjjj'
ANDROID=os.path.join(ROOT,'phone-agent')

def run(cmd, **kwargs):
    print('▶️', ' '.join(map(str,cmd)))
    return subprocess.run(cmd, check=True, **kwargs)

try:
    if os.path.exists(ROOT): shutil.rmtree(ROOT, ignore_errors=True)

    print('1/5 ⬇️ تنزيل المشروع...')
    clone=subprocess.run(['git','clone','--depth','1',REPO,ROOT],timeout=180)
    if clone.returncode != 0:
        print('⚠️ git clone فشل، استخدام ZIP...')
        zp='/content/Jjjjjjj.zip'
        r=subprocess.run(['wget','-q','-O',zp,ZIP_URL],timeout=180)
        if r.returncode != 0: raise RuntimeError('فشل تنزيل ZIP من GitHub')
        with zipfile.ZipFile(zp) as z: z.extractall('/content')
        if os.path.exists(ROOT): shutil.rmtree(ROOT,ignore_errors=True)
        os.rename('/content/Jjjjjjj-main',ROOT)

    if not os.path.isfile(os.path.join(ANDROID,'app','build.gradle.kts')):
        raise RuntimeError('مشروع Android غير موجود في phone-agent/app/')

    print('2/5 ☕ تجهيز Java...')
    candidates=[
        os.environ.get('JAVA_HOME'),
        '/usr/lib/jvm/java-17-openjdk-amd64',
        '/usr/lib/jvm/java-21-openjdk-amd64',
        '/usr/lib/jvm/default-java'
    ]
    java=None
    for p in candidates:
        if p and os.path.isfile(os.path.join(p,'bin','java')) and os.path.isfile(os.path.join(p,'bin','javac')):
            java=p; break

    if java is None:
        print('☕ تثبيت OpenJDK 17...')
        r=subprocess.run(['apt-get','update','-qq'],timeout=240)
        if r.returncode != 0: print('⚠️ apt update exit:',r.returncode)
        r=subprocess.run(['apt-get','install','-y','-qq','openjdk-17-jdk','wget','unzip'],timeout=300)
        if r.returncode != 0: print('⚠️ apt install exit:',r.returncode)
        java='/usr/lib/jvm/java-17-openjdk-amd64'

    if not (os.path.isfile(os.path.join(java,'bin','java')) and os.path.isfile(os.path.join(java,'bin','javac'))):
        raise RuntimeError('لم يتم العثور على JDK صالح في Colab')

    os.environ['JAVA_HOME']=java
    os.environ['PATH']=java+'/bin:'+os.environ.get('PATH','')
    print('JAVA_HOME:',java)

    # لا تستخدم check=True مع java -version؛ بعض توزيعات Java تكتب الإصدار إلى stderr.
    jt=subprocess.run(['java','-version'],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
    print(jt.stdout)
    if jt.returncode != 0: raise RuntimeError('java غير صالح')

    jct=subprocess.run(['javac','-version'],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
    print(jct.stdout)
    if jct.returncode != 0: raise RuntimeError('javac غير صالح')

    print('3/5 📦 تجهيز Gradle...')
    gradle_bin='/content/gradle-8.7/bin/gradle'
    if not os.path.isfile(gradle_bin):
        gz='/content/gradle-8.7-bin.zip'
        r=subprocess.run(['wget','-q','https://services.gradle.org/distributions/gradle-8.7-bin.zip','-O',gz],timeout=300)
        if r.returncode != 0: raise RuntimeError('فشل تنزيل Gradle 8.7')
        with zipfile.ZipFile(gz) as z: z.extractall('/content')

    if not os.path.isfile(gradle_bin): raise RuntimeError('Gradle 8.7 غير موجود')
    os.chmod(gradle_bin,os.stat(gradle_bin).st_mode|stat.S_IXUSR|stat.S_IXGRP|stat.S_IXOTH)
    gv=subprocess.run([gradle_bin,'--version'],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,timeout=120)
    print(gv.stdout[:4000])
    if gv.returncode != 0: raise RuntimeError('Gradle لا يعمل')

    print('4/5 🔨 بناء APK...')
    build_log='/content/pocket_otc_gradle_build.log'
    with open(build_log,'w',encoding='utf-8') as log:
        build=subprocess.run([gradle_bin,'assembleDebug','--no-daemon','--stacktrace'],cwd=ANDROID,stdout=log,stderr=subprocess.STDOUT,text=True,timeout=1200)

    with open(build_log,'r',encoding='utf-8',errors='replace') as f: content=f.read()
    print('\n========== آخر سجل Gradle ==========', '\n')
    print(content[-12000:])
    if build.returncode != 0: raise RuntimeError(f'Gradle فشل — exit code {build.returncode}')

    apk=os.path.join(ANDROID,'app','build','outputs','apk','debug','app-debug.apk')
    print('5/5 📱 التحقق من APK...')
    if not os.path.isfile(apk) or os.path.getsize(apk)==0: raise RuntimeError('لم يتم إنشاء APK')
    print(f'✅ APK: {apk}')
    print(f'📦 الحجم: {os.path.getsize(apk)/1024/1024:.2f} MB')

    from google.colab import files
    files.download(apk)
    print('✅ تم إنشاء APK بنجاح — بدون Telegram — Android 8+ / Android 10 مدعوم.')
except Exception:
    print('❌ فشل البناء:')
    traceback.print_exc()